# Advanced Network Anomaly Detection

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n_samples = 5000
normal_data = np.random.multivariate_normal([100, 50, 20, 80], [[100, 10, 5, 15], [10, 50, 8, 12], [5, 8, 30, 6], [15, 12, 6, 40]], n_samples)
anomaly_data = np.random.multivariate_normal([200, 150, 5, 200], [[200, 50, 20, 30], [50, 100, 15, 25], [20, 15, 60, 10], [30, 25, 10, 80]], 250)
X = np.vstack([normal_data, anomaly_data])
y_true = np.hstack([np.zeros(n_samples), np.ones(250)])

df = pd.DataFrame(X, columns=['packet_size', 'connection_duration', 'failed_logins', 'data_transfer'])
df['is_anomaly'] = y_true

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso_forest = IsolationForest(contamination=0.1, random_state=42, n_estimators=200)
anomaly_pred = iso_forest.fit_predict(X_scaled)
anomaly_scores = iso_forest.decision_function(X_scaled)

df['anomaly_score'] = anomaly_scores
df['predicted_anomaly'] = (anomaly_pred == -1).astype(int)

detection_rate = np.sum((df['is_anomaly'] == 1) & (df['predicted_anomaly'] == 1)) / np.sum(df['is_anomaly'] == 1)
false_positive_rate = np.sum((df['is_anomaly'] == 0) & (df['predicted_anomaly'] == 1)) / np.sum(df['is_anomaly'] == 0)

plt.figure(figsize=(15, 10))
plt.subplot(2, 3, 1)
plt.scatter(df['packet_size'], df['connection_duration'], c=df['predicted_anomaly'], cmap='coolwarm', alpha=0.6)
plt.title('Anomaly Detection: Packet Size vs Connection Duration')
plt.xlabel('Packet Size'); plt.ylabel('Connection Duration')

plt.subplot(2, 3, 2)
plt.hist(df[df['predicted_anomaly'] == 0]['anomaly_score'], bins=50, alpha=0.7, label='Normal', color='blue')
plt.hist(df[df['predicted_anomaly'] == 1]['anomaly_score'], bins=50, alpha=0.7, label='Anomaly', color='red')
plt.title('Anomaly Score Distribution'); plt.xlabel('Anomaly Score'); plt.ylabel('Frequency'); plt.legend()

plt.subplot(2, 3, 3)
correlation_matrix = df[['packet_size', 'connection_duration', 'failed_logins', 'data_transfer']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')

plt.tight_layout(); plt.show()

print(f"Detection Rate: {detection_rate:.3f}")
print(f"False Positive Rate: {false_positive_rate:.3f}")
print(f"Total Anomalies Detected: {np.sum(df['predicted_anomaly'])}")
print(f"Anomaly Score Range: [{df['anomaly_score'].min():.3f}, {df['anomaly_score'].max():.3f}]")